<a href="https://colab.research.google.com/github/pradhapmoorthi/CVND/blob/Trial/2_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Computer Vision Nanodegree

## Project: Image Captioning

---

In this notebook, you will train your CNN-RNN model.  

You are welcome and encouraged to try out many different architectures and hyperparameters when searching for a good model.

This does have the potential to make the project quite messy!  Before submitting your project, make sure that you clean up:
- the code you write in this notebook.  The notebook should describe how to train a single CNN-RNN architecture, corresponding to your final choice of hyperparameters.  You should structure the notebook so that the reviewer can replicate your results by running the code in this notebook.  
- the output of the code cell in **Step 2**.  The output should show the output obtained when training the model from scratch.

This notebook **will be graded**.  

Feel free to use the links below to navigate the notebook:
- [Step 1](#step1): Training Setup
- [Step 2](#step2): Train your Model
- [Step 3](#step3): (Optional) Validate your Model

<a id='step1'></a>
## Step 1: Training Setup

In this step of the notebook, you will customize the training of your CNN-RNN model by specifying hyperparameters and setting other options that are important to the training procedure.  The values you set now will be used when training your model in **Step 2** below.

You should only amend blocks of code that are preceded by a `TODO` statement.  **Any code blocks that are not preceded by a `TODO` statement should not be modified**.

### Task #1

Begin by setting the following variables:
- `batch_size` - the batch size of each training batch.  It is the number of image-caption pairs used to amend the model weights in each training step.
- `vocab_threshold` - the minimum word count threshold.  Note that a larger threshold will result in a smaller vocabulary, whereas a smaller threshold will include rarer words and result in a larger vocabulary.  
- `vocab_from_file` - a Boolean that decides whether to load the vocabulary from file.
- `embed_size` - the dimensionality of the image and word embeddings.  
- `hidden_size` - the number of features in the hidden state of the RNN decoder.  
- `num_epochs` - the number of epochs to train the model.  We recommend that you set `num_epochs=3`, but feel free to increase or decrease this number as you wish.  [This paper](https://arxiv.org/pdf/1502.03044.pdf) trained a captioning model on a single state-of-the-art GPU for 3 days, but you'll soon see that you can get reasonable results in a matter of a few hours!  (_But of course, if you want your model to compete with current research, you will have to train for much longer._)
- `save_every` - determines how often to save the model weights.  We recommend that you set `save_every=1`, to save the model weights after each epoch.  This way, after the `i`th epoch, the encoder and decoder weights will be saved in the `models/` folder as `encoder-i.pkl` and `decoder-i.pkl`, respectively.
- `print_every` - determines how often to print the batch loss to the Jupyter notebook while training.  Note that you **will not** observe a monotonic decrease in the loss function while training - this is perfectly fine and completely expected!  You are encouraged to keep this at its default value of `100` to avoid clogging the notebook, but feel free to change it.
- `log_file` - the name of the text file containing - for every step - how the loss and perplexity evolved during training.

If you're not sure where to begin to set some of the values above, you can peruse [this paper](https://arxiv.org/pdf/1502.03044.pdf) and [this paper](https://arxiv.org/pdf/1411.4555.pdf) for useful guidance!  **To avoid spending too long on this notebook**, you are encouraged to consult these suggested research papers to obtain a strong initial guess for which hyperparameters are likely to work best.  Then, train a single model, and proceed to the next notebook (**3_Inference.ipynb**).  If you are unhappy with your performance, you can return to this notebook to tweak the hyperparameters (and/or the architecture in **model.py**) and re-train your model.

### Question 1

**Question:** Describe your CNN-RNN architecture in detail.  With this architecture in mind, how did you select the values of the variables in Task 1?  If you consulted a research paper detailing a successful implementation of an image captioning model, please provide the reference.

**Answer:**


### (Optional) Task #2

Note that we have provided a recommended image transform `transform_train` for pre-processing the training images, but you are welcome (and encouraged!) to modify it as you wish.  When modifying this transform, keep in mind that:
- the images in the dataset have varying heights and widths, and
- if using a pre-trained model, you must perform the corresponding appropriate normalization.

### Question 2

**Question:** How did you select the transform in `transform_train`?  If you left the transform at its provided value, why do you think that it is a good choice for your CNN architecture?

**Answer:**

### Task #3

Next, you will specify a Python list containing the learnable parameters of the model.  For instance, if you decide to make all weights in the decoder trainable, but only want to train the weights in the embedding layer of the encoder, then you should set `params` to something like:
```
params = list(decoder.parameters()) + list(encoder.embed.parameters())
```

### Question 3

**Question:** How did you select the trainable parameters of your architecture?  Why do you think this is a good choice?

**Answer:**

### Task #4

Finally, you will select an [optimizer](http://pytorch.org/docs/master/optim.html#torch.optim.Optimizer).

### Question 4

**Question:** How did you select the optimizer used to train your model?

**Answer:**

In [1]:
import os, zipfile, urllib.request
from pathlib import Path
import numpy as np

import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.nn.utils.rnn import pack_padded_sequence
from tqdm import tqdm

import torchvision.transforms as T
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
cocoapi_loc = Path(".")
coco_root = cocoapi_loc / "cocoapi"
img_root = coco_root / "images"
ann_root = coco_root / "annotations"

(img_root).mkdir(parents=True, exist_ok=True)
(ann_root).mkdir(parents=True, exist_ok=True)

urls = {
    "train2014.zip": "http://images.cocodataset.org/zips/train2014.zip",
    "val2014.zip": "http://images.cocodataset.org/zips/val2014.zip",
    "annotations_trainval2014.zip": "http://images.cocodataset.org/annotations/annotations_trainval2014.zip"
}

downloads = coco_root / "downloads"
downloads.mkdir(parents=True, exist_ok=True)

def download_if_missing(url, dst):
    dst = Path(dst)
    if dst.exists():
        print(f"✓ Exists: {dst.name}")
        return
    print(f"Downloading {dst.name} ...")
    urllib.request.urlretrieve(url, dst)
    print(f"Saved -> {dst}")

def unzip_if_missing(zip_path, out_dir, marker_path=None):
    zip_path = Path(zip_path)
    out_dir = Path(out_dir)
    if marker_path is not None and Path(marker_path).exists():
        print(f"✓ Unzip skipped (found): {marker_path}")
        return
    print(f"Unzipping {zip_path.name} ...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)
    print("Done.")

# 1) Download zips
for fname, url in urls.items():
    download_if_missing(url, downloads / fname)

# 2) Unzip train/val images into ./cocoapi/images/
unzip_if_missing(downloads/"train2014.zip", img_root, marker_path=img_root/"train2014")
unzip_if_missing(downloads/"val2014.zip", img_root, marker_path=img_root/"val2014")

# 3) Unzip annotations into ./cocoapi/annotations/
unzip_if_missing(downloads/"annotations_trainval2014.zip", coco_root, marker_path=ann_root/"captions_train2014.json")

print("\nExpected paths check:")
print("train2014 exists:", (img_root/"train2014").exists())
print("captions_train2014 exists:", (ann_root/"captions_train2014.json").exists())

In [ ]:
transform = T.Compose([
    T.Resize(256),
    T.RandomCrop(224),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

In [ ]:
import os
if not os.path.exists('data_loader.py'):
    !wget https://raw.githubusercontent.com/pradhapmoorthi/CVND/Trial/data_loader.py

In [ ]:
if not os.path.exists('vocabulary.py'):
    !wget https://raw.githubusercontent.com/pradhapmoorthi/CVND/Trial/vocabulary.py

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Added to download the specific punkt_tab resource
from data_loader import get_loader

from google.colab import drive
drive.mount('/content/gdrive')

import os
from pathlib import Path

# Define the directory to save models and vocab file
save_dir = Path("/content/gdrive/MyDrive/checkpoints")
save_dir.mkdir(exist_ok=True)

batch_size = 64
vocab_threshold = 5
vocab_file = save_dir / "vocab.pkl" # Changed to save in Google Drive
num_workers = 2

# First run: vocab_from_file=False to build vocab.pkl
# Later runs: vocab_from_file=True for speed/reproducibility
vocab_from_file = False

data_loader = get_loader(transform=transform,
                         mode="train",
                         batch_size=batch_size,
                         vocab_threshold=vocab_threshold,
                         vocab_file=vocab_file,
                         vocab_from_file=vocab_from_file,
                         num_workers=num_workers,
                         cocoapi_loc=str(cocoapi_loc))

vocab = data_loader.dataset.vocab
print("Vocab size:", len(vocab))
print("Special tokens:", vocab.start_word, vocab.end_word, vocab.unk_word)


In [ ]:
print("idx2word[0] =", vocab.idx2word.get(0, None))
print("Does <pad> exist in word2idx? ", "<pad>" in vocab.word2idx)
if "<pad>" in vocab.word2idx:
    print("<pad> index =", vocab.word2idx["<pad>"])
print("<start> index =", vocab.word2idx["<start>"])
print("<end> index   =", vocab.word2idx["<end>"])
print("<unk> index   =", vocab.word2idx["<unk>"])

In [ ]:
import os

# Remove existing model.py if it exists
if os.path.exists('model.py'):
    print('Removing existing model.py...')
    os.remove('model.py')

# Download model.py
print('Downloading model.py...')
!wget https://raw.githubusercontent.com/pradhapmoorthi/CVND/Trial/model.py

# Reload the module in case it was modified externally or already imported
import importlib
import model
importlib.reload(model)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from pathlib import Path # Added for Path object

from model import EncoderCNN, DecoderRNN
from data_loader import get_loader
from vocabulary import Vocabulary

# -------------------------
# Configuration
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

embed_size = 512
attention_dim = 512
hidden_size = 512
dropout = 0.5

num_epochs = 10
batch_size = 64
learning_rate = 3e-4

train_images = "cocoapi/images/train2014"
train_captions = "cocoapi/annotations/captions_train2014.json"

# Define the save directory to Google Drive, consistent with earlier setup
save_dir = Path("/content/gdrive/MyDrive/checkpoints")
save_dir.mkdir(exist_ok=True) # Ensure directory exists for saving

# -------------------------
# Data loader
# -------------------------
train_loader, vocab = get_loader(
    root=train_images,
    json=train_captions,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
)

# -------------------------
# Models
# -------------------------
encoder = EncoderCNN(encoded_image_size=7).to(device)
decoder = DecoderRNN(
    attention_dim=attention_dim,
    embed_size=embed_size,
    hidden_size=hidden_size,
    vocab_size=len(vocab),
    encoder_dim=encoder.encoder_dim,
    dropout=dropout,
).to(device)

# -------------------------
# Loss and optimizer
# -------------------------
criterion = nn.CrossEntropyLoss(ignore_index=vocab.word2idx["<pad>"])
params = list(decoder.parameters()) + list(
    p for p in encoder.parameters() if p.requires_grad
)
optimizer = optim.Adam(params, lr=learning_rate)

# -------------------------
# Training loop
# -------------------------
for epoch in range(num_epochs):
    encoder.train()
    decoder.train()

    for images, captions in train_loader:
        images = images.to(device)
        captions = captions.to(device)

        optimizer.zero_grad()

        features = encoder(images)
        outputs, _ = decoder(features, captions)

        loss = criterion(
            outputs.reshape(-1, outputs.size(2)),
            captions[:, 1:].reshape(-1),
        )

        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {loss.item():.4f}")

# -------------------------
# Save checkpoint
# -------------------------
torch.save(
    {
        "encoder": encoder.state_dict(),
        "decoder": decoder.state_dict(),
        "vocab": vocab,
    },
    str(save_dir / "caption_model.pth"), # Modified to save to Google Drive
)

print(f"✅ Training complete and checkpoint saved to {save_dir / 'caption_model.pth'}.")

<a id='step2'></a>
## Step 2: Train your Model

Once you have executed the code cell in **Step 1**, the training procedure below should run without issue.  

It is completely fine to leave the code cell below as-is without modifications to train your model.  However, if you would like to modify the code used to train the model below, you must ensure that your changes are easily parsed by your reviewer.  In other words, make sure to provide appropriate comments to describe how your code works!  

You may find it useful to load saved weights to resume training.  In that case, note the names of the files containing the encoder and decoder weights that you'd like to load (`encoder_file` and `decoder_file`).  Then you can load the weights by using the lines below:

```python
# Load pre-trained weights before resuming training.
encoder.load_state_dict(torch.load(os.path.join('./models', encoder_file)))
decoder.load_state_dict(torch.load(os.path.join('./models', decoder_file)))
```

While trying out parameters, make sure to take extensive notes and record the settings that you used in your various training runs.  In particular, you don't want to encounter a situation where you've trained a model for several hours but can't remember what settings you used :).

### A Note on Tuning Hyperparameters

To figure out how well your model is doing, you can look at how the training loss and perplexity evolve during training - and for the purposes of this project, you are encouraged to amend the hyperparameters based on this information.  

However, this will not tell you if your model is overfitting to the training data, and, unfortunately, overfitting is a problem that is commonly encountered when training image captioning models.  

For this project, you need not worry about overfitting. **This project does not have strict requirements regarding the performance of your model**, and you just need to demonstrate that your model has learned **_something_** when you generate captions on the test data.  For now, we strongly encourage you to train your model for the suggested 3 epochs without worrying about performance; then, you should immediately transition to the next notebook in the sequence (**3_Inference.ipynb**) to see how your model performs on the test data.  If your model needs to be changed, you can come back to this notebook, amend hyperparameters (if necessary), and re-train the model.

That said, if you would like to go above and beyond in this project, you can read about some approaches to minimizing overfitting in section 4.3.1 of [this paper](http://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=7505636).  In the next (optional) step of this notebook, we provide some guidance for assessing the performance on the validation dataset.

<a id='step3'></a>
## Step 3: (Optional) Validate your Model

To assess potential overfitting, one approach is to assess performance on a validation set.  If you decide to do this **optional** task, you are required to first complete all of the steps in the next notebook in the sequence (**3_Inference.ipynb**); as part of that notebook, you will write and test code (specifically, the `sample` method in the `DecoderRNN` class) that uses your RNN decoder to generate captions.  That code will prove incredibly useful here.

If you decide to validate your model, please do not edit the data loader in **data_loader.py**.  Instead, create a new file named **data_loader_val.py** containing the code for obtaining the data loader for the validation data.  You can access:
- the validation images at filepath `'/opt/cocoapi/images/train2014/'`, and
- the validation image caption annotation file at filepath `'/opt/cocoapi/annotations/captions_val2014.json'`.

The suggested approach to validating your model involves creating a json file such as [this one](https://github.com/cocodataset/cocoapi/blob/master/results/captions_val2014_fakecap_results.json) containing your model's predicted captions for the validation images.  Then, you can write your own script or use one that you [find online](https://github.com/tylin/coco-caption) to calculate the BLEU score of your model.  You can read more about the BLEU score, along with other evaluation metrics (such as TEOR and Cider) in section 4.1 of [this paper](https://arxiv.org/pdf/1411.4555.pdf).  For more information about how to use the annotation file, check out the [website](http://cocodataset.org/#download) for the COCO dataset.